# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/umarfarukh786/FlyRank-task1/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

For the local starter contract, **one row is one pseudonymized content item**. The row contains trailing-90-day aggregate activity and derived rates, plus two adjacent comparison windows: the most recent 30 days and the preceding 30 days. The file is a cross-sectional export, so it has no per-row report date; the exact export date is not recoverable from the CSV alone. The warehouse contract is different: its daily fact grain is one `report_date × client_id × content_id` row, with dates spanning 2025-01-27 through 2026-06-30.

In [13]:
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent.parent
RAW_PATH = ROOT / "data" / "raw" / "content_refresh_anonymized.csv"
if not RAW_PATH.exists():
    raise FileNotFoundError(f"Starter dataset not found: {RAW_PATH}")

df = pd.read_csv(RAW_PATH)
required_columns = {"content_id", "client_id", "impressions_90d", "trend_direction"}
missing_columns = required_columns.difference(df.columns)
if missing_columns:
    raise ValueError(f"Missing required columns: {sorted(missing_columns)}")

print(f"Loaded {len(df):,} rows and {len(df.columns):,} columns")
print(f"Unique content items: {df['content_id'].nunique():,}")
print(f"Unique pseudonymized clients: {df['client_id'].nunique():,}")
print("Declared windows: trailing 90d; last 30d; previous 30d; no row-level date column")
assert df["content_id"].is_unique
print("Grain check on the starter slice: passed")

Loaded 30,000 rows and 44 columns
Unique content items: 30,000
Unique pseudonymized clients: 32
Declared windows: trailing 90d; last 30d; previous 30d; no row-level date column
Grain check on the starter slice: passed


## 2. Fields: feature / label / context / excluded

The planned refresh-ranking analysis uses only fields knowable in the snapshot before a review decision. `trend_direction` and `trend_pct` are label sources, not features. IDs support grouping and joins only. The current/previous 30-day comparison fields are retained for label construction and audit, but excluded from model features when the outcome is derived from that comparison.

In [14]:
feature_numeric = [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "impressions_90d", "clicks_90d", "pageviews_90d", "sessions_90d",
    "users_90d", "engaged_sessions_90d", "ai_sessions_90d", "scroll_events_90d",
    "days_with_impressions", "days_with_sessions", "content_age_days",
    "days_since_last_update", "ctr", "avg_position", "engagement_rate",
    "scroll_rate", "ai_traffic_pct",
]
feature_categorical = [
    "competition_level", "content_type", "main_intent", "age_tier",
    "freshness_tier", "word_count_tier", "impression_tier", "position_tier",
]
label_fields = ["trend_direction", "trend_pct", "is_declining_label"]
context_fields = ["content_id", "client_id"]
excluded_fields = {
    "provider_used": "generation metadata is not part of the decision signal",
    "model_used": "generation metadata is not part of the decision signal",
    "impressions_last_30d": "overlaps the current trend-label window",
    "clicks_last_30d": "overlaps the current trend-label window",
    "sessions_last_30d": "overlaps the current trend-label window",
    "impressions_prev_30d": "used in the trend comparison, not a model feature for this proxy label",
    "clicks_prev_30d": "used in the trend comparison, not a model feature for this proxy label",
    "sessions_prev_30d": "used in the trend comparison, not a model feature for this proxy label",
}

field_contract = pd.DataFrame([
    *[{"field": field, "bucket": "feature", "why": "snapshot signal available before review"} for field in feature_numeric],
    *[{"field": field, "bucket": "feature", "why": "categorical snapshot context"} for field in feature_categorical],
    *[{"field": field, "bucket": "label / proxy", "why": "defines or is derived into the observed decline label"} for field in label_fields],
    *[{"field": field, "bucket": "context", "why": "grouping, joining, or client-holdout splitting only"} for field in context_fields],
    *[{"field": field, "bucket": "excluded", "why": why} for field, why in excluded_fields.items()],
])
display(field_contract)
all_bucketed = set(feature_numeric + feature_categorical + label_fields + context_fields + list(excluded_fields))
assert len(all_bucketed) == len(field_contract)
assert not set(label_fields).intersection(feature_numeric + feature_categorical)
assert not set(context_fields).intersection(feature_numeric + feature_categorical)
print(f"Contracted fields: {len(field_contract)}")
print("Label, context, and excluded fields are disjoint from features: passed")

,field,bucket,why
0,search_volume,feature,snapshot signal available before review
1,competition,feature,snapshot signal available before review
2,cpc,feature,snapshot signal available before review
3,word_count,feature,snapshot signal available before review
4,char_count,feature,snapshot signal available before review
5,impressions_90d,feature,snapshot signal available before review
6,clicks_90d,feature,snapshot signal available before review
7,pageviews_90d,feature,snapshot signal available before review
8,sessions_90d,feature,snapshot signal available before review
9,users_90d,feature,snapshot signal available before review


Contracted fields: 43
Label, context, and excluded fields are disjoint from features: passed


## 3. Verify it with queries (grain, counts, missing values, windows)

The checks below are pandas equivalents of the contract queries for the local CSV. They verify the promised grain and counts, inspect patterned missingness by `content_type`, and confirm the available 90-day/30-day window columns. The local snapshot has no date column, so a date-range query is not possible here; the warehouse date range is recorded in Section 1 from the release documentation.

In [15]:
duplicate_content = (
    df.groupby("content_id", dropna=False).size().reset_index(name="rows")
)
duplicate_content = duplicate_content[duplicate_content["rows"] > 1]
print(f"Duplicate content IDs: {len(duplicate_content):,}")
assert duplicate_content.empty

print("Rows by content type:")
print(df["content_type"].value_counts(dropna=False).to_string())

missingness = df[["search_volume", "competition", "cpc", "word_count", "char_count"]].isna().mean().sort_values(ascending=False)
print("Overall missingness for keyword/content fields:")
print(missingness.round(3).to_string())

missing_by_type = df.groupby("content_type", dropna=False)[
    ["search_volume", "word_count", "char_count"]
].apply(lambda group: group.isna().mean())
print("Missingness by content type:")
print(missing_by_type.round(3).to_string())

zero_position_rate = (pd.to_numeric(df["avg_position"], errors="coerce") == 0).mean()
no_previous_impressions = (pd.to_numeric(df["impressions_prev_30d"], errors="coerce") == 0).mean()
print(f"avg_position == 0 (no position data): {zero_position_rate:.3f}")
print(f"impressions_prev_30d == 0: {no_previous_impressions:.3f}")

expected_window_columns = {
    "impressions_90d", "clicks_90d", "sessions_90d",
    "impressions_last_30d", "clicks_last_30d", "sessions_last_30d",
    "impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d",
}
assert expected_window_columns.issubset(df.columns)
print("Window-column check: passed")

Duplicate content IDs: 0
Rows by content type:
content_type
keyword article       27207
feedly article         2096
comparison article      697
Overall missingness for keyword/content fields:
char_count       0.257
word_count       0.257
search_volume    0.082
cpc              0.082
competition      0.082
Missingness by content type:
                    search_volume  word_count  char_count
content_type                                             
comparison article          0.000       0.000       0.000
feedly article              1.000       0.000       0.000
keyword article             0.014       0.283       0.283
avg_position == 0 (no position data): 0.040
impressions_prev_30d == 0: 0.113
Window-column check: passed


## 4. Data limits

This starter contract cannot identify the exact export date because the CSV has no row-level date field. It also cannot distinguish a causal refresh effect from an observed trend association. The local slice is one cross-sectional snapshot, not the full warehouse panel: it cannot test future-period generalization, client history depth, or time-aware performance.

The warehouse adds important limits: client histories are unbalanced; early GA4/GSC values can be zero-filled when availability flags are false or null; the fixed query table window can overlap a future label; and repeated per-content query context must be aggregated with `ANY_VALUE`, not summed. These limits must be addressed before extending this contract to warehouse-scale work.

In [16]:
limits = pd.DataFrame([
    {
        "limit": "No row-level export date in starter CSV",
        "consequence": "Exact calendar window cannot be independently verified from this file.",
    },
    {
        "limit": "Trend label is derived from the current snapshot comparison",
        "consequence": "Use as an observed proxy, not as a future causal outcome.",
    },
    {
        "limit": "Cross-sectional starter slice",
        "consequence": "Cannot establish future-period drift or intervention impact.",
    },
    {
        "limit": "Structured missingness by content type",
        "consequence": "Do not interpret blank keyword/content fields as random missingness or blindly fill them with zero.",
    },
    {
        "limit": "Warehouse panel and query-window caveats",
        "consequence": "Require per-client history checks, availability flags, and strict feature/label window alignment.",
    },
])
display(limits)
print("Contract conclusion: valid for a public-safe starter-slice ranking workflow with the stated limitations.")

,limit,consequence
0,No row-level export date in starter CSV,Exact calendar window cannot be independently ...
1,Trend label is derived from the current snapsh...,"Use as an observed proxy, not as a future caus..."
2,Cross-sectional starter slice,Cannot establish future-period drift or interv...
3,Structured missingness by content type,Do not interpret blank keyword/content fields ...
4,Warehouse panel and query-window caveats,"Require per-client history checks, availabilit..."


Contract conclusion: valid for a public-safe starter-slice ranking workflow with the stated limitations.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.